---
title: "Reddit Results Extraction"
format:
  html:
    embed-resources: true
    code-fold: true
    toc: true
---

## Overview

This section covers the methodology of how we extracted our Reddit results after all the data had been processed and labelled in previous steps. It showcases the process of aggregation to obtain the final "Reddit thermometer" results, which will be contrasted against official polls.

## Code & Process

For this step, we used DuckDB to read the data and perform the operations, due to its speed processing large datasets utilizing SQL queries. In the script below, you can see how we read the labelled data, aggregated by month and year, and then counted the number of time each label appears per month. This created our counts for Democrat, Republican, and No Party Preference.

In [ ]:
import sys
!{sys.executable} -m pip install duckdb
import duckdb
import pandas as pd
from azureml.core import Workspace, Datastore, Dataset

#workspace set up and identification
#dummy variables for subscription_id
subscription_id = 'MY_SUBSCRIPTION_ID' # Removed for security concerns
resource_group = 'project-group-11'
workspace_name = 'project-group-11'

workspace = Workspace(subscription_id, resource_group, workspace_name)

datastore = Datastore.get(workspace, "workspaceblobstore")

#reading in both the comments & submissions
dataset_submissions = Dataset.Tabular.from_parquet_files(path=(datastore, 'cleandata/submissions_labeled_Jude.parquet'))
dataset_comments = Dataset.Tabular.from_parquet_files(path=(datastore, 'cleandata/comments_labeled_Jude.parquet'))

# Convert Tabular Datasets to Pandas DataFrames
df_submissions = dataset_submissions.to_pandas_dataframe()
df_comments = dataset_comments.to_pandas_dataframe()

# Use DuckDB to process data
con = duckdb.connect()

# Query to combine and count Party occurrences by Month and Year
query = """
SELECT 
    year, 
    month, 
    Party, 
    COUNT(*) AS party_count
FROM (
    SELECT year, month, Party FROM df_submissions
    UNION ALL
    SELECT year, month, Party FROM df_comments
)
GROUP BY year, month, Party
ORDER BY year, month, Party;
"""

# Execute the query and get the result
result = con.execute(query).fetchdf()

## Post Processing Review

This is how the data looks after processing:

In [ ]:
from azureml.core import Dataset
from azureml.data.datapath import DataPath

target_path = DataPath(datastore, "cleandata/counts_for_presentation")

result_tabular = Dataset.Tabular.register_pandas_dataframe(
    dataframe=result,
    target=target_path,
    name="counts_for_presentation",
    description="Party counts grouped by month and year",
    show_progress=True
)

result.head()

,year,month,Party,party_count
0,2022,1,Democrat,11388
1,2022,1,No Party,861962
2,2022,1,Republican,12636
3,2022,2,Democrat,10654
4,2022,2,No Party,820484


## Last Thoughts Before Results

As you can see above, our labelled dataset contains a great deal of posts categorized as "No Party" where our labelling logic could not deduce the party lean of the post in question. While this is interesting in and of itself, and was retained in the data for the purposes of training or models in during the Machine Learning section, we discarded No Party Preference posts for the purpose of analyzing our findings. These posts often fell into one of three camps: discussing the keywords in a nonpolitical way, having a neutral sentiment, or having too many keywords from both the left and right to make a determination with confidence. As such, we did not feel that they would be helpful in fully grasping the mood of those Redditors who were discussing partisan politics in a clearly identifiable way.

Now that we have our aggregated Reddit data, we can start to plot it and draw some conclusions.